# Step 4b: answer at the right level of detail

Scores hierarchical answers ("animal", then "bird", then "owl") against flat answers, using the
predictions saved by step 4a. **Nothing is trained, so no GPU is needed.** The tree and every comparison
were frozen beforehand in `configs/hierarchy_v1.yaml` and `configs/hierarchy_eval_v1.yaml`.

**Before running:**
1. In **Input**, the `AI_pictionary` notebook output must be the **step 4a version** (it holds
   `runs/anytime_v1/anytime/*.npz`). If the notebook stops with "predictions not found", remove that input
   and add it again so Kaggle attaches the latest version.
2. Keep your Kaggle Dataset attached and **Internet on**.
3. Choose **Accelerator: None**, then **Save Version → Save & Run All**.


In [ ]:
import os, shutil, subprocess
from pathlib import Path

REPO_URL = "https://github.com/Trung4n/AI-dictionary.git"
BRANCH = "pipeline-step1"
REPO = Path("/kaggle/working/repo")

token = None
try:
    from kaggle_secrets import UserSecretsClient
    token = UserSecretsClient().get_secret("GITHUB_TOKEN")
except Exception:
    print("No GITHUB_TOKEN secret: cloning without one (works for a public repository).")

url = REPO_URL.replace("https://", f"https://x-access-token:{token}@") if token else REPO_URL
os.chdir("/kaggle/working")  # a re-run may still be inside the folder deleted next
if REPO.exists():
    shutil.rmtree(REPO)
done = subprocess.run(["git", "clone", "--depth", "1", "-b", BRANCH, url, str(REPO)], capture_output=True, text=True)
if done.returncode != 0:
    message = done.stderr.replace(token, "***") if token else done.stderr
    raise RuntimeError(f"clone failed:\n{message}")
# Keep the token out of files that Kaggle may save with the notebook output.
subprocess.run(["git", "-C", str(REPO), "remote", "set-url", "origin", REPO_URL], check=True)
print(subprocess.run(["git", "-C", str(REPO), "log", "-1", "--oneline"], capture_output=True, text=True).stdout)


In [ ]:
%cd /kaggle/working/repo
!pip install -q pytest
tests = subprocess.run("PYTHONPATH=src python -m pytest tests -q -k 'not processed_data'", shell=True, capture_output=True, text=True)
print(tests.stdout[-1500:])
if tests.returncode != 0:
    raise RuntimeError("tests failed")

found = [p.parent for p in Path("/kaggle/input").rglob("gru_prefix_s42.npz")]
if not found:
    raise RuntimeError("predictions not found: attach the step 4a version of the AI_pictionary output")
PREDICTIONS = found[0]
print("predictions:", PREDICTIONS, sorted(p.name for p in PREDICTIONS.glob("*.npz")))


In [ ]:
!PYTHONPATH=src python -m quickdraw_exp.hierarchy --matrix configs/hierarchy_eval_v1.yaml --predictions {PREDICTIONS}
# Rescore step 4a from the same saved predictions with the threshold fix (float64 scores); no model is run.
!PYTHONPATH=src python -m quickdraw_exp.anytime --matrix configs/anytime_v1.yaml --predictions {PREDICTIONS} --out /kaggle/working/runs/anytime_v1_rescored


In [ ]:
import json

result = json.loads(Path("/kaggle/working/runs/hierarchy_v1/results.json").read_text())
print("missing runs:", result["missing_runs"])
print("\nPre-registered comparisons (reward area, evaluation half):")
for row in result["comparisons"]:
    if row["difference"] is None:
        print(f"  [{row['precision_target']}] {row['name']}: not available")
    else:
        print(f"  [{row['precision_target']}] {row['name']}: {row['a_mean']:.4f} vs {row['b_mean']:.4f}, "
              f"difference {row['difference']:+.4f}  (a {[round(v, 4) for v in row['a_values']]}, b {[round(v, 4) for v in row['b_values']]})")
print("\nPer run (precision target 0.95):")
for run, by_target in result["per_run"].items():
    for tree, r in by_target["0.95"].items():
        print(f"  {run:16s} {tree:9s} tau {r['tau']}  area {r['reward_area']:.4f}  @0.25 {r['reward_at_0.25']:.4f}  "
              f"@0.5 {r['reward_at_0.5']:.4f}  precision {r['precision']:.4f}  answered {r['answered']:.3f}  levels {r['answers_by_level']}")

rescored = json.loads(Path("/kaggle/working/runs/anytime_v1_rescored/results.json").read_text())
print("\nStep 4a rescored (threshold fix), risk control on the evaluation half:")
for condition, policies in rescored["table"].items():
    for policy, e in policies.items():
        if "earliness_at_gap_0.02" in e:
            print(f"  {condition:20s} {policy:7s} auc {e['auc']['mean']:.4f}  earliness@gap2% {e['earliness_at_gap_0.02']['mean']:.3f} "
                  f"met {e['met_gap_0.02']}  earliness@gap5% {e['earliness_at_gap_0.05']['mean']:.3f} met {e['met_gap_0.05']}")
